This notebook shows how to use LLM-Geo, you can uncomment a study case (preferred Case 1 and Case 3) to get the spatial analysis results without any human intervention. 

Please check the [webpage of LLM-Geo](https://github.com/gladcolor/LLM-Geo) for more detail!


#  Install package

Make sure you are using the latest version of `openai` and `geopandas`.

In [1]:
# ! pip install pyvis
# ! pip install networkx
# ! pip install openai
# ! pip install openai --upgrade
# ! conda update  --channel conda-forge geopandas  


# Import package

In [1]:
import os
import sys
import requests
import networkx as nx
import pandas as pd
import geopandas as gpd
import rasterio
from pyvis.network import Network
from openai import OpenAI
from IPython.display import display, HTML, Code
from IPython.display import clear_output

# Define Solution class
Please run the following cell to define the functions

In [2]:
%load_ext autoreload
%autoreload 2

import LLM_Geo_Constants as constants
import helper

from LLM_Geo_kernel import Solution

sys.path.append(os.path.abspath("Modules"))    
import Modules.data_eye as data_eye

# Demonstration Cases

## Input task and data desciption

In [3]:
isReview = True

In [4]:

# Case 1: population living near hazardous wastes

task_name ='Resident_at_risk_counting'
TASK = r"""1) Find out Census tracts that contain hazardous waste facilities, then compute and print out the population living in those tracts. The study area is North Carolina (NC), US.
2) Generate a population choropleth map for all tract polygons in NC, rendering the color by tract population; and then highlight the borders of tracts that have hazardous waste facilities. Please draw all polygons, not only the highlighted ones. The map size is 15*10 inches.
"""

# data location with column information
DATA_LOCATIONS = ["NC hazardous waste facility ESRI shape file: https://github.com/gladcolor/LLM-Geo/raw/master/overlay_analysis/HW_Sites_EPSG4326.zip.",
                  "NC tract boundary shapefile: https://github.com/gladcolor/LLM-Geo/raw/master/overlay_analysis/tract_37_EPSG4326.zip. The tract ID column is 'GEOID', data types is integer.",
                  "NC tract population CSV file: https://github.com/gladcolor/LLM-Geo/raw/master/overlay_analysis/NC_tract_population.csv. The population is stored in 'TotalPopulation' column. The tract ID column is 'GEOID', data types is integer."
                 ]

# data location without column information
# DATA_LOCATIONS = ["NC hazardous waste facility ESRI shape file: https://github.com/gladcolor/LLM-Geo/raw/master/overlay_analysis/HW_Sites_EPSG4326.zip.",
#                   "NC tract boundary shapefile: https://github.com/gladcolor/LLM-Geo/raw/master/overlay_analysis/tract_37_EPSG4326.zip.",
#                   "NC tract population CSV file: https://github.com/gladcolor/LLM-Geo/raw/master/overlay_analysis/NC_tract_population.csv."
#                  ]
 

# # Case 3: COVID-19 death rate in US
# task_name ='COVID_death_rate'
# TASK = r'''1) Draw a choropleth map to show the death rate (death/case) of COVID-19 among the countiguous US counties. Use the accumulated COVID-19 data of 2020.12.31 to compute the death rate. Use scheme ='quantiles' when plotting the map.  Set map projection to 'Conus Albers'. Set map size to 15*10 inches.  
# 2) Draw a scatter plot to show the correlation and trend line of the death rate with the senior resident rate, including the r-square and p-value. Set data point transparency to 50%, regression line as red. Set figure size to 15*10 inches.  
# '''

# data location with column information
# DATA_LOCATIONS = [
#                   r"COVID-19 data case in 2020 (county-level): https://github.com/nytimes/covid-19-data/raw/master/us-counties-2020.csv. This data is for daily accumulated COVID cases and deaths for each county in the US. There are 5 columns: date (format: 2021-02-01), county, state, fips, cases, deaths. ",   
#                   r"Contiguous US county boundary (ESRI shapefile): https://github.com/gladcolor/spatial_data/raw/master/contiguous_counties.zip. The county FIPS column is 'GEOID'; map projection is EPSG:4269",
#                   r"Census data (ACS2020): https://raw.githubusercontent.com/gladcolor/spatial_data/master/Demography/ACS2020_5year_county.csv. THe needed columns are: 'FIPS', 'Total Population', 'Total Population: 65 to 74 Years', 'Total Population: 75 to 84 Years', 'Total Population: 85 Years and Over'. Drop rows with NaN cells after loading the used columns.",
#                  ]

# data location without column information
# DATA_LOCATIONS = [
#                   r"COVID-19 data case in 2020 (county-level): https://github.com/nytimes/covid-19-data/raw/master/us-counties-2020.csv. This data is for daily accumulated COVID cases and deaths for each county in the US. ",   
#                   r"Contiguous US county boundary (ESRI shapefile): https://github.com/gladcolor/spatial_data/raw/master/contiguous_counties.zip.",
#                   r"Census data (ACS2020): https://raw.githubusercontent.com/gladcolor/spatial_data/master/Demography/ACS2020_5year_county.csv. "
#                  ]

# # Case 4: Hospital_accessibility
# task_name ='Hospital_accessibility'
# TASK = r'''
# For each zipcode area in South Carolina (SC), calculate the distance from the centroid of the zipcode area to its nearest hospital, and then create a choropleth distance map of zipcode area polygons (unit: km), also show the hospital.
# '''

## data location with column information
# DATA_LOCATIONS = [
# r"SC zipcode boundary shapefile: https://github.com/GIBDUSC/test/raw/master/sc_zip_boundary.zip, the map projection is WGS1984.",
# r"SC hospitals:  https://github.com/gladcolor/spatial_data/raw/master/South_Carolina/SC_hospitals_with_emergency_room_cleaned.csv, location columns: longitude in 'Longitude' column, latitude in 'Latitude' column.",          
# ]

# data location without column information
# DATA_LOCATIONS = [
# r"SC zipcode boundary shapefile: https://github.com/GIBDUSC/test/raw/master/sc_zip_boundary.zip.",
# r"SC hospitals:  https://github.com/gladcolor/spatial_data/raw/master/South_Carolina/SC_hospitals_with_emergency_room_cleaned.csv.",          
# ]

# # Case 5: school walkability
# task_name ='School walkability'
# TASK = r'''You need to assess the walkability for all schools in the Colubmia city. Requirements:
# 1) Using 1 km buffer zone as the study area for each school, compuate each school's walkability scores, which is the ratio of the sidewalk length to the road length in the buffer zone.
# 2) Draw a map for each school, using the school name and the walkability score as the map title, while showing the sidewalks and the buffer zone on a OpenStreetMap basemap.
# '''
#
# # data location with column information
# DATA_LOCATIONS = [
# r"Columbia school (points): https://github.com/gladcolor/spatial_data/raw/refs/heads/master/South_Carolina/Columbia_schools.gpkg.",
# r"Columbia road network (polyline): https://github.com/gladcolor/spatial_data/raw/refs/heads/master/South_Carolina/Columbia_road.gpkg.",
# r"Columbia sidewalk network (polyline): https://github.com/gladcolor/spatial_data/raw/refs/heads/master/South_Carolina/Columbia_sidewalks.gpkg."
# ]

##  Case 6: Trunk diameter error
# task_name = "Trunk diameter error"
# TASK = """You need to compute the location error between the estimated tree trunk and the ground truth: 
# 1) for each ground truth location, find out its nearest estimated location within 3 meters. 
# 2) compute the RMSE for ground truth trunks that have the associated estimated location."
# """

# DATA_LOCATIONS = [
# r"Estimated trunk location: D:\OneDrive_PSU\OneDrive - The Pennsylvania State University\Data\Street_view_image\trunk_location_estimated_EPSG6569.shp.", 
# r"Trunk ground truth: D:\OneDrive_PSU\OneDrive - The Pennsylvania State University\Data\Street_view_image\trunk_location_ground_truth_EPSG6569.shp."
# ]

##  Case 7: Population difference
# task_name = "Population difference"
# TASK = """Make a thematic map, show the difference between the resident population data and the LandScan gridded nighttime population for the Census tracts in North Carolina. 
# """
# DATA_LOCATIONS = [
# r"North Carolina Census tract boundaris: D:/OneDrive_PSU/OneDrive - The Pennsylvania State University/Data/LandScan/North_Carolina_tracts.shp", 
# r"USA residential population at block group level: D:/OneDrive_PSU/OneDrive - The Pennsylvania State University/Data/LandScan/Residential_population.csv",
# r"USA LandScan nighttime population at block group level: D:/OneDrive_PSU/OneDrive - The Pennsylvania State University/Data/LandScan/landscan-usa-2021-night.csv"
# ]


##  Case 8: Store site selection
# task_name = "Store site selection"
# TASK = """Propose a new Walmart store location, which has similar distances to the existing four locations, while near the roads within 300 meters.
# """

# DATA_LOCATIONS = [
# r"Existing Walmart locations: D:/OneDrive_PSU/OneDrive - The Pennsylvania State University/Data/Walmart_site/Walmart_4_site_EPSG32111.shp", 
# r"Road network: D:/OneDrive_PSU/OneDrive - The Pennsylvania State University/Data/Walmart_site/Motorway_EPSG32111.shp",
# ]

##  Case 9: random points
# task_name = "Random points"
# TASK = """You have a point layer, and need to do: inside the extent of this layer, generate as much as possible random points that with a minimum distance of 100 m, while with a minimum distance of 30 m to the given points. 
# """

# DATA_LOCATIONS = [
# r"D:\OneDrive_PSU\OneDrive - The Pennsylvania State University\Data\Street_tree\Richland_downtown_street_view_image_location.shp",
# ]

##  Case 10: Air quality sensor
# task_name = "Air_quality_sensor"
# TASK = """The loaded data is the real-time readings of a PM2.5 sensors on 2023-06-29, 341 readings in total. 
# You need to create an HTML report containing: 
# 1) a map showing the location of air quality sensor with  a basemap. 
# 2) a plot showing real-time readings of the sensor. 
# 3) a table showing all the readings.
# """

# DATA_LOCATIONS = [
# r"E:\OneDrive_PSU\OneDrive - The Pennsylvania State University\Data\Air_quality\1289807.csv",
# ]

# # Case 2: France_mobility_changes_2020  (NOTE: invalidated due to API shutdown)
# task_name ='France_mobility_changes_2020'
# TASK = r'''
# 1) Show the 2020 human mobility monthly change rates of each administrative regions in a France choropleth map. Each month is a sub-map in a map matrix，12 months in total. All monthly maps need to use the same colorbar range (color scheme: coolwarm). The base of the change rate is January 2020. 
# 2) Draw a line chart to show the monthly change rate trends of all administrative regeions. Each region is a line (the region name is the legend), the x-axis is 2020 months.
# '''

# DATA_LOCATIONS = ["ESRI shapefile for France administrative regions:" + \
#                   "https://github.com/gladcolor/LLM-Geo/raw/master/REST_API/France.zip. " + \
#                   "The 'GID_1' column is the administrative region code, 'NAME_1' column is the administrative region name.",
#                   "REST API url with parameters for daily human mobility data access:" + \
#                   "http://gis.cas.sc.edu/GeoAnalytics/REST?operation=get_daily_movement_for_all_places&source=twitter&scale=world_first_level_admin&begin=01/01/2020&end=12/31/2020." + \
#                   "The response is in CSV format. There are three columns in the response: " + \
#                   "place,date (format:2020-01-07), and intra_movement. 'place' column is the administractive region code of every country;" + \
#                   "codes for France administrative regions start with 'FRA'. Use the total intra_movement of the month as the montly mobility.",
#                  ]


save_dir = os.path.join(os.getcwd(), task_name)
os.makedirs(save_dir, exist_ok=True)




In [7]:
# create graph
# model = r'gpt-4o'
# model = r'gpt-4o-2024-08-06'
# model = r'o1-preview'  # Not ready yet. 2024-12-26/
# model = r'o1'  # Not ready yet. 2025-02-05/
# model = r'o3-mini'  # Slow, may need to reasoning before generating tokens. Tested on 2025-02-09/
model = r'gpt-6-luna'

# model = r'o1-mini'   #  Error code: 400 - {'error': {'message': "Unsupported value: 'messages[0].role' does not support 'system' with this model.", 'type': 'invalid_request_error', 'param': 'messages[0].role', 'code': 'unsupported_value'}}
# no system prompt,  no temperature setting, no response format (json-object)

In [7]:
# file_path = r'https://github.com/gladcolor/spatial_data/raw/refs/heads/master/Everest_DEM.tif'
# file_path = r'https://github.com/gladcolor/spatial_data/raw/refs/heads/master/Everest_DOM.tif'
# file_path = r"D:\OneDrive_PSU\OneDrive - The Pennsylvania State University\Research_doc\LLM_Geo\QGIS_plugin\sentinel2.tif"
# file_path = r"C:\Users\A\Downloads\Sentinel_downloaded\L1C_T31TBF_A043377_20231012T105229_2023-10-12\L1C_T31TBF_A043377_20231012T105229_B09.jp2"

## Get data overview (column names, data types, and map projection)

In [9]:
attributes_json, DATA_LOCATIONS = data_eye.add_data_overview_to_data_location(task=TASK, data_location_list=DATA_LOCATIONS, model=model)
print("DATA_LOCATIONS with data overviews:")
print(DATA_LOCATIONS)

Error in get_data_overview() {'location': 'https://github.com/gladcolor/LLM-Geo/raw/master/overlay_analysis/HW_Sites_EPSG4326.zip (ESRI shapefile)', 'format': 'ESRI shapefile'} URL can't contain control characters. '/gladcolor/LLM-Geo/raw/master/overlay_analysis/HW_Sites_EPSG4326.zip (ESRI shapefile)' (found at least ' ')
Error in get_data_overview() {'location': 'https://github.com/gladcolor/LLM-Geo/raw/master/overlay_analysis/tract_37_EPSG4326.zip (ESRI shapefile)', 'format': 'ESRI shapefile'} URL can't contain control characters. '/gladcolor/LLM-Geo/raw/master/overlay_analysis/tract_37_EPSG4326.zip (ESRI shapefile)' (found at least ' ')
Error in get_data_overview() {'location': 'https://github.com/gladcolor/LLM-Geo/raw/master/overlay_analysis/NC_tract_population.csv (CSV)', 'format': 'CSV'} 'NoneType' object has no attribute 'dtypes'


KeyError: 'meta_str'

In [9]:
solution = Solution(
                    task=TASK,
                    task_name=task_name,
                    save_dir=save_dir,
                    data_locations=DATA_LOCATIONS,
                    model=model,
                    )
print("Prompt to get solution graph:\n")
print(solution.graph_prompt)

Prompt to get solution graph:

Your role: A professional Geo-information scientist and programmer good at Python. You have worked on Geographic information science more than 20 years, and know every detail and pitfall when processing spatial data and coding. You know well how to set up workflows for spatial analysis tasks. You have significant experence on graph theory, application, and implementation. You are also experienced on generating map using Matplotlib and GeoPandas.
 

Your task: Generate a graph (data structure) only, whose nodes are (1) a series of consecutive steps and (2) data to solve this question:  
 You need to assess the walkability for all schools in the Colubmia city. Requirements:
1) Using 1 km buffer zone as the study area for each school, compuate each school's walkability scores, which is the ratio of the sidewalk length to the road length in the buffer zone.
2) Draw a map for each school, using the school name and the walkability score as the map title, while 

## Get graph code from GPT API

In [10]:
response_for_graph = solution.get_LLM_response_for_graph() 
solution.graph_response = response_for_graph
solution.save_solution()

clear_output(wait=True)
display(Code(solution.code_for_graph, language='python'))

import networkx as nx
G = nx.DiGraph()

# 1. Load Columbia school points
G.add_node("Columbia_schools", node_type="data", data_path="https://github.com/gladcolor/spatial_data/raw/refs/heads/master/South_Carolina/Columbia_schools.gpkg", description="Columbia school points dataset (EPSG:4326)")
G.add_node("load_school_points", node_type="operation", data_path="", description="Load Columbia school points from URL")
G.add_edge("Columbia_schools", "load_school_points")
G.add_node("school_points_gdf", node_type="data", data_path="", description="School points GeoDataFrame")
G.add_edge("load_school_points", "school_points_gdf")

# 2. Load Columbia road network
G.add_node("Columbia_roads", node_type="data", data_path="https://github.com/gladcolor/spatial_data/raw/refs/heads/master/South_Carolina/Columbia_road.gpkg", description="Columbia road network dataset (EPSG:4326)")
G.add_node("load_roads", node_type="operation", data_path="", description="Load Columbia road network from URL")
G.add_edge("Columbia_roads", "load_roads")
G.add_node("roads_gdf", node_type="data", data_path="", description="Road network GeoDataFrame")
G.add_edge("load_roads", "roads_gdf")

# 3. Load Columbia sidewalk network
G.add_node("Columbia_sidewalks", node_type="data", data_path="https://github.com/gladcolor/spatial_data/raw/refs/heads/master/South_Carolina/Columbia_sidewalks.gpkg", description="Columbia sidewalk network dataset (EPSG:4326)")
G.add_node("load_sidewalks", node_type="operation", data_path="", description="Load Columbia sidewalk network from URL")
G.add_edge("Columbia_sidewalks", "load_sidewalks")
G.add_node("sidewalks_gdf", node_type="data", data_path="", description="Sidewalk network GeoDataFrame")
G.add_edge("load_sidewalks", "sidewalks_gdf")

# 4. Generate 1km buffer for each school
G.add_node("generate_school_buffers", node_type="operation", data_path="", description="Buffer each school point by 1 km to create study areas")
G.add_edge("school_points_gdf", "generate_school_buffers")
G.add_node("school_buffers_gdf", node_type="data", data_path="", description="GeoDataFrame of school buffers (1 km)")
G.add_edge("generate_school_buffers", "school_buffers_gdf")

# 5. Extract roads within each school buffer
G.add_node("extract_roads_in_buffer", node_type="operation", data_path="", description="Spatially extract road segments intersecting each school buffer")
G.add_edge("roads_gdf", "extract_roads_in_buffer")
G.add_edge("school_buffers_gdf", "extract_roads_in_buffer")
G.add_node("roads_within_buffers", node_type="data", data_path="", description="Road segments within each school buffer")
G.add_edge("extract_roads_in_buffer", "roads_within_buffers")

# 6. Extract sidewalks within each school buffer
G.add_node("extract_sidewalks_in_buffer", node_type="operation", data_path="", description="Spatially extract sidewalk segments intersecting each school buffer")
G.add_edge("sidewalks_gdf", "extract_sidewalks_in_buffer")
G.add_edge("school_buffers_gdf", "extract_sidewalks_in_buffer")
G.add_node("sidewalks_within_buffers", node_type="data", data_path="", description="Sidewalk segments within each school buffer")
G.add_edge("extract_sidewalks_in_buffer", "sidewalks_within_buffers")

# 7. Compute total road length within each buffer per school
G.add_node("compute_roads_length", node_type="operation", data_path="", description="Compute total road length within each school buffer")
G.add_edge("roads_within_buffers", "compute_roads_length")
G.add_node("roads_length_by_school", node_type="data", data_path="", description="Total road length per school buffer")
G.add_edge("compute_roads_length", "roads_length_by_school")

# 8. Compute total sidewalk length within each buffer per school
G.add_node("compute_sidewalks_length", node_type="operation", data_path="", description="Compute total sidewalk length within each school buffer")
G.add_edge("sidewalks_within_buffers", "compute_sidewalks_length")
G.add_node("sidewalks_length_by_school", node_type="data", data_path=

## Execute code to generate the solution graph

In [11]:
exec(solution.code_for_graph)
solution_graph = solution.load_graph_file()

# Show the graph
G = nx.read_graphml(solution.graph_file)  
nt = helper.show_graph(G)
html_name = os.path.join(os.getcwd(), solution.task_name + '.html')  
# HTML file should in the same directory. See:
# https://stackoverflow.com/questions/65564916/error-displaying-pyvis-html-inside-jupyter-lab-cell
nt.show(name=html_name)
# html_name

 

C:\Users\u6025895\Documents\GitHub\LLM-Geo\School walkability.html


## Generate prompts and code for operations (functions)

In [12]:
operations = solution.get_LLM_responses_for_operations(review=isReview)
solution.save_solution()

all_operation_code_str = '\n'.join([operation['operation_code'] for operation in operations])

clear_output(wait=True)
display(Code(all_operation_code_str, language='python'))

import geopandas as gpd

def load_school_points(Columbia_schools='https://github.com/gladcolor/spatial_data/raw/refs/heads/master/South_Carolina/Columbia_schools.gpkg'):
    """
    Load Columbia school points from URL
    Columbia_schools: URL to the Columbia school points dataset (GeoPackage file, EPSG:4326)
    """
    # Load the school points GeoDataFrame using GeoPandas
    school_points_gdf = gpd.read_file(Columbia_schools)
    
    # Drop records with missing values in the 'name' column (critical for further processing)
    school_points_gdf = school_points_gdf.dropna(subset=['name'])
    
    # Ensure the GeoDataFrame is in the correct CRS (EPSG:4326)
    school_points_gdf = school_points_gdf.to_crs(epsg=4326)
    
    return school_points_gdf
import geopandas as gpd

def load_roads(Columbia_roads='https://github.com/gladcolor/spatial_data/raw/refs/heads/master/South_Carolina/Columbia_road.gpkg'):
    """
    Load Columbia road network from URL.
    
    Parameters:
        Columbia_roads (str): URL to the Columbia road network GeoPackage.
        
    Returns:
        roads_gdf (GeoDataFrame): Road network GeoDataFrame in EPSG:4326 projection.
    """
    # Load the road network using geopandas
    roads_gdf = gpd.read_file(Columbia_roads)
    
    # Drop records with missing geometries to ensure reliable spatial processing
    roads_gdf = roads_gdf.dropna(subset=['geometry'])
    
    # Ensure the road network is in EPSG:4326 projection; if not, convert it
    if roads_gdf.crs is None or roads_gdf.crs.to_string() != "EPSG:4326":
        roads_gdf = roads_gdf.to_crs("EPSG:4326")
    
    return roads_gdf
import geopandas as gpd

def load_sidewalks(Columbia_sidewalks='https://github.com/gladcolor/spatial_data/raw/refs/heads/master/South_Carolina/Columbia_sidewalks.gpkg'):
    """
    Load Columbia sidewalk network from URL.
    
    Parameters:
    Columbia_sidewalks (str): URL to the Columbia sidewalk network GeoPackage.
    
    Returns:
    sidewalks_gdf (GeoDataFrame): GeoDataFrame containing the Columbia sidewalk network,
                                  with NaN geometries dropped.
    """
    # Load the sidewalk network GeoPackage using GeoPandas
    sidewalks_gdf = gpd.read_file(Columbia_sidewalks)
    
    # Drop any features with missing geometries to ensure robust further processing
    sidewalks_gdf = sidewalks_gdf.dropna(subset=['geometry'])
    
    return sidewalks_gdf
def generate_school_buffers(school_points_gdf):
    """
    Buffer each school point by 1 km to create study areas.
    """
    import geopandas as gpd

    # Drop records with missing geometry to avoid errors during buffering
    school_points_gdf = school_points_gdf.dropna(subset=['geometry'])
    
    # Ensure the input GeoDataFrame is in EPSG:4326 before projecting
    if school_points_gdf.crs.to_string() != "EPSG:4326":
        school_points_gdf = school_points_gdf.to_crs(epsg=4326)
    
    # Estimate an appropriate UTM CRS for metric buffer calculations
    target_crs = school_points_gdf.estimate_utm_crs()
    points_proj = school_points_gdf.to_crs(target_crs)
    
    # Buffer each school point by 1000 meters (1 km)
    points_proj["geometry"] = points_proj.geometry.buffer(1000)
    
    # Reproject the buffered geometries back to EPSG:4326 for consistency with other data layers
    school_buffers_gdf = points_proj.to_crs(epsg=4326)
    
    return school_buffers_gdf
def extract_roads_in_buffer(roads_gdf=roads_gdf, school_buffers_gdf=school_buffers_gdf):
    """
    Spatially extract road segments intersecting each school buffer.
    
    Parameters:
        roads_gdf (GeoDataFrame): Columbia road network GeoDataFrame in EPSG:4326.
        school_buffers_gdf (GeoDataFrame): GeoDataFrame of school buffers (1 km) in EPSG:4326.
    
    Returns:
        roads_within_buffers (GeoDataFrame): Road segments that intersect each school buffer.
    """
    import geopandas as gpd

    # Drop records with missing geometries
    roads_gdf = r

## Generate prompts and code for assembly program

In [13]:
assembly_LLM_response = solution.get_LLM_assembly_response(review=isReview)
solution.assembly_LLM_response = assembly_LLM_response
solution.save_solution()

clear_output(wait=True)
display(Code(solution.code_for_assembly, language='python'))

import os
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import contextily as ctx

def load_school_points(Columbia_schools='https://github.com/gladcolor/spatial_data/raw/refs/heads/master/South_Carolina/Columbia_schools.gpkg'):
    """
    Load Columbia school points from URL
    Columbia_schools: URL to the Columbia school points dataset (GeoPackage file, EPSG:4326)
    """
    school_points_gdf = gpd.read_file(Columbia_schools)
    school_points_gdf = school_points_gdf.dropna(subset=['name'])
    school_points_gdf = school_points_gdf.to_crs(epsg=4326)
    return school_points_gdf

def load_roads(Columbia_roads='https://github.com/gladcolor/spatial_data/raw/refs/heads/master/South_Carolina/Columbia_road.gpkg'):
    """
    Load Columbia road network from URL.
    """
    roads_gdf = gpd.read_file(Columbia_roads)
    roads_gdf = roads_gdf.dropna(subset=['geometry'])
    if roads_gdf.crs is None or roads_gdf.crs.to_string() != "EPSG:4326":
        roads_gdf = roads_gdf.to_crs("EPSG:4326")
    return roads_gdf

def load_sidewalks(Columbia_sidewalks='https://github.com/gladcolor/spatial_data/raw/refs/heads/master/South_Carolina/Columbia_sidewalks.gpkg'):
    """
    Load Columbia sidewalk network from URL.
    """
    sidewalks_gdf = gpd.read_file(Columbia_sidewalks)
    sidewalks_gdf = sidewalks_gdf.dropna(subset=['geometry'])
    return sidewalks_gdf

def generate_school_buffers(school_points_gdf):
    """
    Buffer each school point by 1 km to create study areas.
    """
    school_points_gdf = school_points_gdf.dropna(subset=['geometry'])
    if school_points_gdf.crs.to_string() != "EPSG:4326":
        school_points_gdf = school_points_gdf.to_crs(epsg=4326)
    target_crs = school_points_gdf.estimate_utm_crs()
    points_proj = school_points_gdf.to_crs(target_crs)
    points_proj["geometry"] = points_proj.geometry.buffer(1000)
    school_buffers_gdf = points_proj.to_crs(epsg=4326)
    return school_buffers_gdf

def extract_roads_in_buffer(roads_gdf, school_buffers_gdf):
    """
    Spatially extract road segments intersecting each school buffer.
    """
    roads_gdf = roads_gdf.dropna(subset=['geometry'])
    school_buffers_gdf = school_buffers_gdf.dropna(subset=['geometry'])
    if roads_gdf.crs != school_buffers_gdf.crs:
        roads_gdf = roads_gdf.to_crs(school_buffers_gdf.crs)
    roads_within_buffers = gpd.sjoin(
        roads_gdf,
        school_buffers_gdf,
        how='inner',
        predicate='intersects',
        lsuffix='road',
        rsuffix='buffer'
    )
    return roads_within_buffers

def extract_sidewalks_in_buffer(sidewalks_gdf, school_buffers_gdf):
    """
    Spatially extract sidewalk segments intersecting each school buffer.
    """
    if sidewalks_gdf.crs != school_buffers_gdf.crs:
        sidewalks_gdf = sidewalks_gdf.to_crs(school_buffers_gdf.crs)
    sidewalks_within_buffers = gpd.sjoin(
        sidewalks_gdf, 
        school_buffers_gdf, 
        how="inner", 
        predicate="intersects", 
        lsuffix="sidewalk", 
        rsuffix="buffer"
    )
    sidewalks_within_buffers = sidewalks_within_buffers.drop_duplicates(subset=['id', 'geometry'])
    return sidewalks_within_buffers

def compute_roads_length(roads_within_buffers):
    """
    Compute total road length within each school buffer.
    """
    roads_within_buffers = roads_within_buffers.dropna(subset=['geometry', 'index_right'])
    if roads_within_buffers.crs is None:
        raise ValueError("roads_within_buffers does not have a CRS defined.")
    metric_crs = roads_within_buffers.estimate_utm_crs()
    roads_metric = roads_within_buffers.to_crs(metric_crs)
    roads_metric["length_m"] = roads_metric.geometry.length
    length_by_school = roads_metric.groupby("index_right")["length_m"].sum().reset_index()
    roads_length_by_school = length_by_school.rename(
        columns={"index_right": "school_buffer_index", "length_m": "total_road_length_m"}
    )
    return roads_length_by_

## Execute assembly code

In [15]:
all_code = all_operation_code_str + '\n' + solution.code_for_assembly

# display(Code(all_code, language='python'))

all_code = solution.execute_complete_program(code=all_code, try_cnt=10)



-------------- Running code (trial # 1/10) --------------


Error_info_str: 
Traceback (most recent call last):
  File "Complete program", line 85, in <module>
    def extract_roads_in_buffer(roads_gdf=roads_gdf, school_buffers_gdf=school_buffers_gdf):
NameError: name 'roads_gdf' is not defined

Sending error information to LLM for debugging...
Below is the entire corrected program with the revisions explained below:

• Removed the default arguments (e.g., roads_gdf=roads_gdf) from the definition of extract_roads_in_buffer because those variables were not defined in the global scope. This fixes the NameError.

• Kept the overall program structure as required and ensured that all spatial layers are reprojected when needed.

• Ensured use of gpd.read_file(URL) for zipped ESRI shapefiles (in our case using GeoPackage URLs) and confirmation that no pandas.StringIO is used.

Below is the complete corrected code:

```python
import os
import geopandas as gpd
import pandas as pd
import numpy